# Code-Switch Dataset + Evaluation Pipeline

This notebook is designed for Google Colab with GPU enabled. It builds the three datasets and evaluates four LLMs on them.

## 1. Runtime Setup

In Colab, switch to `Runtime -> Change runtime type -> GPU` before running the cells below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf codeswitch-project
!cp -r '/content/drive/MyDrive/CodeSwitch Lig Project' codeswitch-project
%cd /content/codeswitch-project
!pip install -q -r requirements.txt

## 2. Optional Hugging Face Login

Run this if you want to use the default Llama models from the config.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Inspect The Local Input Files

In [ ]:
!head -n 3 dataset/spa.txt
!head -n 3 dataset/spanish_texts/lid_spaeng_train.csv

## 4. Generate The Control Sample

This creates `control_eng.csv` from 300 sampled MultiWOZ user prompts.

In [ ]:
NUM_SAMPLES = 300
%env PYTHONPATH=/content/codeswitch-project/src
!python scripts/run_pipeline.py --stage sample --config configs/pipeline.yaml --num-samples $NUM_SAMPLES

## 5. Build `unfinetuned_engesp.csv` and `finetuned_engesp.csv`

This stage also writes the candidate audit CSVs and the LoRA adapter.

In [ ]:
!python scripts/run_pipeline.py --stage datasets --config configs/pipeline.yaml --num-samples $NUM_SAMPLES

## 6. Evaluate Four LLMs

This writes `llm_eval_raw.csv` and `llm_eval_summary.csv`.

In [ ]:
!python scripts/run_pipeline.py --stage evaluate --config configs/pipeline.yaml --num-samples $NUM_SAMPLES

## 7. Preview The Outputs

In [ ]:
import pandas as pd

control_df = pd.read_csv('outputs/datasets/control_eng.csv')
unfinetuned_df = pd.read_csv('outputs/datasets/unfinetuned_engesp.csv')
finetuned_df = pd.read_csv('outputs/datasets/finetuned_engesp.csv')
summary_df = pd.read_csv('outputs/evaluations/llm_eval_summary.csv')

display(control_df.head())
display(unfinetuned_df.head())
display(finetuned_df.head())
display(summary_df)

## 8. Copy Results Back To Drive

In [ ]:
!cp -r outputs '/content/drive/MyDrive/CodeSwitch Lig Project/'